# 07 -- System Integration & Final Report
**ITAI 2373 -- Final Project | Leroy Brown**

This notebook wires every module together in a single end-to-end pipeline run,
exports all results to `data/results/`, and generates the comprehensive project summary report.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings
warnings.filterwarnings('ignore')

import json, time
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import kagglehub

from config.settings import (
    DATASET_SIZE, RANDOM_STATE, N_TOPICS, SEMANTIC_INDEX_SIZE,
    SUMMARIZER_MODEL, EMBEDDING_MODEL
)
from src.data_processing.preprocessor import preprocess_text
from src.data_processing.data_loader import load_bbc_dataset
from src.data_processing.data_validator import validate_dataframe, clean_dataframe
from src.analysis.classifier import NewsClassifier
from src.analysis.topic_modeler import TopicModeler
from src.analysis.sentiment_analyzer import score_text
from src.language_models.summarizer import summarize_with_stats
from src.language_models.embeddings import SemanticSearchIndex
from src.multilingual.translator import translate_and_detect
from src.conversation.query_processor import QueryProcessor
from src.conversation.intent_classifier import classify_intent

RESULTS = os.path.join("..", "data", "results")
os.makedirs(RESULTS, exist_ok=True)

print("All imports OK.")
print(f"Results directory: {os.path.abspath(RESULTS)}")

## 1 -- Full Pipeline Execution

Every stage is timed and its outputs captured for the final report.

In [ ]:
pipeline_log = {}
t0_total = time.time()

# Stage 1: Data loading and validation
t0 = time.time()
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
df_raw = load_bbc_dataset(path)
val_report = validate_dataframe(df_raw)
df = clean_dataframe(df_raw).sample(min(DATASET_SIZE, len(df_raw)), random_state=RANDOM_STATE).reset_index(drop=True)
pipeline_log["stage1_data"] = {
    "raw_articles":       len(df_raw),
    "clean_articles":     len(df),
    "categories":         df["category"].nunique(),
    "passed_validation":  val_report["passed"],
    "time_s":             round(time.time() - t0, 2),
}
print(f"[Stage 1] {len(df):,} articles loaded in {pipeline_log['stage1_data']['time_s']}s")

In [ ]:
# Stage 2: Preprocessing
t0 = time.time()
df["clean_text"] = df["text"].apply(preprocess_text)
df["word_count"]  = df["text"].apply(lambda t: len(t.split()))
pipeline_log["stage2_preprocess"] = {
    "avg_words_before": round(df["word_count"].mean(), 1),
    "avg_clean_tokens": round(df["clean_text"].apply(lambda t: len(t.split())).mean(), 1),
    "time_s":           round(time.time() - t0, 2),
}
print(f"[Stage 2] Done in {pipeline_log['stage2_preprocess']['time_s']}s")

In [ ]:
# Stage 3: TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
vectorizer = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=2)
X_tfidf = vectorizer.fit_transform(df["clean_text"])
pipeline_log["stage3_tfidf"] = {
    "features":  X_tfidf.shape[1],
    "documents": X_tfidf.shape[0],
    "time_s":    round(time.time() - t0, 2),
}
print(f"[Stage 3] TF-IDF {X_tfidf.shape} in {pipeline_log['stage3_tfidf']['time_s']}s")

In [ ]:
# Stage 4: Classification
t0 = time.time()
clf = NewsClassifier(vectorizer=vectorizer)
clf_results = clf.train_test_evaluate(X_tfidf, df["category"])
pipeline_log["stage4_classification"] = {
    "accuracy": round(clf_results["accuracy"], 4),
    "time_s":   round(time.time() - t0, 2),
}
print(f"[Stage 4] Accuracy: {clf_results['accuracy']:.1%}  ({pipeline_log['stage4_classification']['time_s']}s)")

In [ ]:
# Stage 5: Topic modeling (LDA + NMF)
from sklearn.feature_extraction.text import CountVectorizer

t0 = time.time()
count_vec = CountVectorizer(max_features=5_000, min_df=2, stop_words="english")
dtm = count_vec.fit_transform(df["clean_text"])

lda_model = TopicModeler(n_topics=N_TOPICS, method="lda")
lda_dist  = lda_model.fit_transform(dtm, count_vec)
lda_words = {i: lda_model.get_topic_words(i) for i in range(N_TOPICS)}

nmf_model = TopicModeler(n_topics=N_TOPICS, method="nmf")
nmf_dist  = nmf_model.fit_transform(dtm, count_vec)
nmf_words = {i: nmf_model.get_topic_words(i) for i in range(N_TOPICS)}

pipeline_log["stage5_topics"] = {
    "n_topics":         N_TOPICS,
    "lda_top_words_t0": lda_words[0],
    "nmf_top_words_t0": nmf_words[0],
    "time_s":           round(time.time() - t0, 2),
}
print(f"[Stage 5] LDA + NMF fitted in {pipeline_log['stage5_topics']['time_s']}s")

In [ ]:
# Stage 6: Sentiment
t0 = time.time()
sentiments = df["text"].apply(score_text)
df["compound"] = sentiments.apply(lambda s: s["sent_compound"])
df["sent_pos"] = sentiments.apply(lambda s: s["sent_pos"])
df["sent_neg"] = sentiments.apply(lambda s: s["sent_neg"])
pipeline_log["stage6_sentiment"] = {
    "mean_compound": round(df["compound"].mean(), 4),
    "pct_positive":  round((df["compound"] >= 0.05).mean() * 100, 1),
    "pct_negative":  round((df["compound"] <= -0.05).mean() * 100, 1),
    "pct_neutral":   round(((df["compound"] > -0.05) & (df["compound"] < 0.05)).mean() * 100, 1),
    "by_category":   df.groupby("category")["compound"].mean().round(4).to_dict(),
    "time_s":        round(time.time() - t0, 2),
}
print(f"[Stage 6] Mean compound: {pipeline_log['stage6_sentiment']['mean_compound']}  ({pipeline_log['stage6_sentiment']['time_s']}s)")

In [ ]:
# Stage 7: Summarization (one sample per category)
t0 = time.time()
sample_summaries = {}
for cat in df["category"].unique():
    art = df[df["category"] == cat]["text"].iloc[0]
    stats = summarize_with_stats(art)
    sample_summaries[cat] = {
        "original_words":  stats["original_words"],
        "summary_words":   stats["summary_words"],
        "compression_pct": stats["compression_pct"],
        "preview":         stats["summary"][:120],
    }
avg_comp = round(sum(v['compression_pct'] for v in sample_summaries.values()) / len(sample_summaries), 1)
pipeline_log["stage7_summarization"] = {
    "model":               SUMMARIZER_MODEL,
    "avg_compression_pct": avg_comp,
    "time_s":              round(time.time() - t0, 2),
}
print(f"[Stage 7] Avg compression: {avg_comp}%  ({pipeline_log['stage7_summarization']['time_s']}s)")

In [ ]:
# Stage 8: Semantic search index
t0 = time.time()
search_index = SemanticSearchIndex()
search_index.build(df)
embedding_dim = search_index.embeddings.shape[1] if hasattr(search_index, 'embeddings') else 384
pipeline_log["stage8_search"] = {
    "model":         EMBEDDING_MODEL,
    "index_size":    len(search_index.index),
    "embedding_dim": embedding_dim,
    "time_s":        round(time.time() - t0, 2),
}
print(f"[Stage 8] Index: {pipeline_log['stage8_search']['index_size']} articles  ({pipeline_log['stage8_search']['time_s']}s)")

In [ ]:
# Stage 9: Multilingual
t0 = time.time()
test_texts = {
    "es": "El presidente anuncio nuevas reformas economicas para el proximo ano.",
    "fr": "Le gouvernement a presente son nouveau plan de relance economique.",
    "de": "Die Bundesregierung plant umfangreiche Investitionen in erneuerbare Energien.",
    "pt": "O banco central anunciou uma nova politica de juros.",
}
multi_results = {}
for lang, text in test_texts.items():
    res = translate_and_detect(text)
    multi_results[lang] = {
        "detected":   res["source_lang"],
        "translated": res["translated"][:100],
        "error":      res.get("error"),
    }
all_correct = all(r['detected'] == lang for lang, r in multi_results.items())
pipeline_log["stage9_multilingual"] = {
    "languages_tested":       list(test_texts.keys()),
    "all_detected_correctly":  all_correct,
    "time_s":                 round(time.time() - t0, 2),
}
print(f"[Stage 9] Detection correct: {all_correct}  ({pipeline_log['stage9_multilingual']['time_s']}s)")

In [ ]:
# Stage 10: Conversational interface
t0 = time.time()
stats_cache = {
    "total_articles":  len(df),
    "accuracy":        clf_results["accuracy"],
    "category_counts": df["category"].value_counts().to_dict(),
    "avg_sentiment":   pipeline_log["stage6_sentiment"]["by_category"],
    "avg_words":       df.groupby("category")["word_count"].mean().round(0).astype(int).to_dict(),
    "top_topics":      {cat: lda_words[i % N_TOPICS][:5]
                        for i, cat in enumerate(df["category"].unique())},
}
qp = QueryProcessor(
    classifier=clf, vectorizer=vectorizer, preprocessor=preprocess_text,
    summarizer_fn=summarize_with_stats, search_index=search_index,
    translator_fn=translate_and_detect, sentiment_fn=score_text,
    stats_cache=stats_cache,
)
intent_tests = [
    ("help",                  "help"),
    ("stats",                 "stats"),
    ("search: AI technology", "search"),
    ("what are the topics?",  "topics"),
]
intent_ok = sum(1 for q, exp in intent_tests if classify_intent(q) == exp)
pipeline_log["stage10_conversation"] = {
    "intents_tested":  len(intent_tests),
    "intents_correct": intent_ok,
    "time_s":          round(time.time() - t0, 2),
}
pipeline_log['total_time_s'] = round(time.time() - t0_total, 2)
print(f"[Stage 10] Intents: {intent_ok}/{len(intent_tests)}  ({pipeline_log['stage10_conversation']['time_s']}s)")
print(f"\nTotal pipeline time: {pipeline_log['total_time_s']}s")

## 2 -- Results Export

In [ ]:
RESULTS = os.path.join("..", "data", "results")
os.makedirs(RESULTS, exist_ok=True)

with open(os.path.join(RESULTS, 'pipeline_log.json'), 'w') as f:
    json.dump(pipeline_log, f, indent=2)

with open(os.path.join(RESULTS, 'sample_summaries.json'), 'w') as f:
    json.dump(sample_summaries, f, indent=2)

with open(os.path.join(RESULTS, 'multilingual_results.json'), 'w') as f:
    json.dump(multi_results, f, indent=2)

with open(os.path.join(RESULTS, 'topic_words.json'), 'w') as f:
    json.dump({'lda': lda_words, 'nmf': nmf_words}, f, indent=2)

df[['category', 'word_count', 'compound']].to_csv(
    os.path.join(RESULTS, 'article_scores.csv'), index=False)

print('All results exported:')
for fname in os.listdir(RESULTS):
    size = os.path.getsize(os.path.join(RESULTS, fname))
    print(f'  {fname}  ({size:,} bytes)')

## 3 -- System Performance Dashboard

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('NewsBot 2.0 -- System Performance Dashboard', fontsize=16, fontweight='bold')

cats = list(stats_cache['category_counts'].keys())
counts = [stats_cache['category_counts'][c] for c in cats]
sent_vals = [pipeline_log['stage6_sentiment']['by_category'].get(c, 0) for c in cats]

# Category distribution
axes[0, 0].bar(cats, counts, color=sns.color_palette('Set2', len(cats)))
axes[0, 0].set_title('Article Distribution by Category')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=30)

# Sentiment by category
colors = ['#2ecc71' if v >= 0.05 else '#e74c3c' if v <= -0.05 else '#95a5a6' for v in sent_vals]
axes[0, 1].bar(cats, sent_vals, color=colors)
axes[0, 1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0, 1].set_title('Average Sentiment by Category')
axes[0, 1].set_ylabel('VADER Compound Score')
axes[0, 1].tick_params(axis='x', rotation=30)

# Pipeline timing
stage_keys = ['data','preprocess','tfidf','classification','topics',
              'sentiment','summarization','search','multilingual','conversation']
stage_labels = [f'S{i+1}' for i in range(len(stage_keys))]
times = [pipeline_log.get(f'stage{i+1}_{k}', {}).get('time_s', 0)
         for i, k in enumerate(stage_keys)]
axes[0, 2].barh(stage_labels, times, color='#3498db')
axes[0, 2].set_title('Stage Execution Time (seconds)')
axes[0, 2].set_xlabel('Seconds')

# Sentiment distribution
axes[1, 0].hist(df['compound'], bins=40, color='#9b59b6', edgecolor='white')
axes[1, 0].axvline(0.05,  color='#2ecc71', linestyle='--', label='Positive threshold')
axes[1, 0].axvline(-0.05, color='#e74c3c', linestyle='--', label='Negative threshold')
axes[1, 0].set_title('Sentiment Score Distribution')
axes[1, 0].set_xlabel('VADER Compound')
axes[1, 0].legend(fontsize=8)

# Compression by category
comp_vals = [sample_summaries.get(c, {}).get('compression_pct', 0) for c in cats]
axes[1, 1].bar(cats, comp_vals, color=sns.color_palette('Set1', len(cats)))
axes[1, 1].set_title('Summarization Compression by Category')
axes[1, 1].set_ylabel('Compression %')
axes[1, 1].tick_params(axis='x', rotation=30)

# Word count distribution
for cat in cats:
    subset = df[df['category'] == cat]['word_count']
    axes[1, 2].hist(subset, bins=30, alpha=0.5, label=cat)
axes[1, 2].set_title('Word Count Distribution by Category')
axes[1, 2].set_xlabel('Words')
axes[1, 2].legend(fontsize=7)

plt.tight_layout()
dash_path = os.path.join(RESULTS, 'system_dashboard.png')
plt.savefig(dash_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Dashboard saved: {dash_path}')

## 4 -- Confusion Matrix -- Final Classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

_, X_test, _, y_test = train_test_split(
    X_tfidf, df['category'], test_size=0.2, random_state=RANDOM_STATE, stratify=df['category']
)
y_pred = clf.model.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=cats)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=cats, yticklabels=cats, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f"Confusion Matrix -- Accuracy: {pipeline_log['stage4_classification']['accuracy']:.1%}")
plt.tight_layout()
cm_path = os.path.join(RESULTS, 'confusion_matrix_final.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_test, y_pred))
print(f'Saved: {cm_path}')

## 5 -- Final Project Report

In [ ]:
s4  = pipeline_log['stage4_classification']
s6  = pipeline_log['stage6_sentiment']
s7  = pipeline_log['stage7_summarization']
s8  = pipeline_log['stage8_search']
s9  = pipeline_log['stage9_multilingual']
s10 = pipeline_log['stage10_conversation']

lines = [
    '=' * 70,
    '  NewsBot 2.0 -- ITAI 2373 Final Project Report',
    '  Leroy Brown | Houston Community College',
    '=' * 70,
    '',
    'DATASET',
    f"  Source      : BBC News Archive (Kaggle)",
    f"  Articles    : {pipeline_log['stage1_data']['raw_articles']:,} raw -> {pipeline_log['stage1_data']['clean_articles']:,} cleaned",
    f"  Categories  : {pipeline_log['stage1_data']['categories']}",
    f"  Validation  : {'PASSED' if pipeline_log['stage1_data']['passed_validation'] else 'FAILED'}",
    '',
    'CLASSIFICATION',
    f"  Algorithm   : Logistic Regression (max_iter=1000)",
    f"  Features    : {pipeline_log['stage3_tfidf']['features']:,}",
    f"  Accuracy    : {s4['accuracy']:.1%}",
    '',
    'TOPIC MODELING',
    f"  Methods     : LDA + NMF  |  Topics: {N_TOPICS}",
    f"  LDA Topic 0 : {', '.join(lda_words[0][:5])}",
    f"  NMF Topic 0 : {', '.join(nmf_words[0][:5])}",
    '',
    'SENTIMENT',
    f"  Mean score  : {s6['mean_compound']}  |  Pos: {s6['pct_positive']}%  Neg: {s6['pct_negative']}%  Neu: {s6['pct_neutral']}%",
    '',
    'SUMMARIZATION',
    f"  Model       : {s7['model']}",
    f"  Compression : {s7['avg_compression_pct']}% average",
    '',
    'SEMANTIC SEARCH',
    f"  Model       : {s8['model']}  |  Index: {s8['index_size']} articles  |  Dims: {s8['embedding_dim']}",
    '',
    'MULTILINGUAL',
    f"  Languages   : {', '.join(s9['languages_tested'])}",
    f"  Detection   : {'All correct' if s9['all_detected_correctly'] else 'Errors'}",
    '',
    'CONVERSATIONAL INTERFACE',
    f"  Intents     : 7  |  Verified: {s10['intents_correct']}/{s10['intents_tested']}",
    '',
    f"TOTAL PIPELINE TIME : {pipeline_log['total_time_s']}s",
    '=' * 70,
]
print('\n'.join(lines))

In [ ]:
report_path = os.path.join(RESULTS, 'final_report.txt')
with open(report_path, 'w') as f:
    f.write('\n'.join(lines))
print(f'Report saved: {report_path}')
print('\nAll results exported successfully.')

## 6 -- Rubric Compliance Checklist

| Requirement | Implementation | Status |
|-------------|---------------|--------|
| Data preprocessing pipeline | `src/data_processing/` | PASS |
| Text classification (LR) | `NewsClassifier` -- TF-IDF + LogisticRegression, 97%+ | PASS |
| Topic modeling (LDA + NMF) | `TopicModeler(method='lda'/'nmf')` | PASS |
| Abstractive summarization | DistilBART `sshleifer/distilbart-cnn-12-6` | PASS |
| Semantic search | Sentence-BERT `all-MiniLM-L6-v2`, cosine similarity | PASS |
| Sentiment analysis | VADER multi-dimension scoring | PASS |
| Named entity recognition | spaCy `en_core_web_sm` | PASS |
| Multilingual support | langdetect + deep-translator | PASS |
| Conversational interface | Gradio ChatInterface, 7 intents | PASS |
| Modular codebase | `src/` subpackages with `__init__.py` | PASS |
| Unit + integration tests | pytest in `tests/` | PASS |
| BBC News Archive dataset | 2,225 articles, 5 categories | PASS |
| Results exported | `data/results/` -- JSON, CSV, PNG | PASS |

**All rubric requirements satisfied.**